# Uzun fonksiyonlar icin ek AI uretimi (deepseek-coder 6.7B)

Ucretsiz Kaggle GPU'sunda `scripts/02b_generate_long.py gen --shard 1/2` calistirir
(isin diger yarisi yerel bilgisayarda `--shard 0/2` ile yapilir). Gereken tek girdi:
`long_prompts.jsonl` - sag panelden veri seti olarak eklenir.
Cikti: `/kaggle/working/ai_long_kaggle.jsonl` ve `ai_long_kaggle_failed.jsonl`.

In [ ]:
# 1) Ollama kurulumu ve model
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                 stderr=subprocess.STDOUT)
time.sleep(15)
!ollama --version
!ollama pull deepseek-coder:6.7b-instruct
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Kod (GitHub) ve istem dosyasi
import glob, shutil
!rm -rf /tmp/tez && git clone --depth 1 https://github.com/MetinSerinkaya27/tez-ai-kod-tespiti.git /tmp/tez
found = glob.glob("/kaggle/input/**/long_prompts.jsonl", recursive=True)
assert found, "long_prompts.jsonl bulunamadi: sag panelden 'Add Input' ile yukleyin"
os.makedirs("/tmp/tez/data/interim", exist_ok=True)
shutil.copy(found[0], "/tmp/tez/data/interim/long_prompts.jsonl")
# Onceki bir calismanin ciktisi varsa kaldigi yerden devam etmek icin geri kopyala
for name in ("ai_long_kaggle.jsonl", "ai_long_kaggle_failed.jsonl"):
    prev = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if prev:
        shutil.copy(prev[0], f"/tmp/tez/data/interim/{name}")
n = sum(1 for _ in open("/tmp/tez/data/interim/long_prompts.jsonl", encoding="utf-8"))
print("istem:", n)
!pip install -q uv

In [ ]:
# 3) Uretim (tahmini 6-7 saat). Her ornek bitince diske yazilir; 5 dakikada bir
# ciktilar /kaggle/working'e kopyalanir ki sure dolarsa elde kalanlar indirilebilsin.
%cd /tmp/tez
import subprocess, shutil, threading, time
def yedekle():
    while True:
        for n in ("ai_long_kaggle.jsonl", "ai_long_kaggle_failed.jsonl"):
            p = f"/tmp/tez/data/interim/{n}"
            if os.path.exists(p):
                shutil.copy(p, f"/kaggle/working/{n}")
        time.sleep(300)
threading.Thread(target=yedekle, daemon=True).start()
!PYTHONPATH=src uv run --no-project --python 3.12 --with requests --with tqdm python -u scripts/02b_generate_long.py gen --shard 1/2 --out data/interim/ai_long_kaggle.jsonl
!cp data/interim/ai_long_kaggle.jsonl data/interim/ai_long_kaggle_failed.jsonl /kaggle/working/
!wc -l /kaggle/working/ai_long_kaggle*.jsonl